# 01: Build a weather data pipeline on AWS

This notebook moves a small NOAA weather sample through three layers:

```text
NOAA public S3 bucket -> your raw S3 prefix -> cleaned Parquet -> Glue Data Catalog
```

You will choose the sample, find a bad source value, clean it, set weather thresholds, make a validation check fail, and compare CSV with Parquet. AWS resource creation and Glue table definitions are marked as provided code.

Expected time: 55 minutes.

In SageMaker, attach `AmazonAthenaFullAccess` to the execution role. When running locally, complete `aws login` before starting Jupyter; the signed-in principal needs equivalent S3, Glue, and Athena permissions.

## 1. Set up the notebook

The project keeps local copies so you can inspect the raw CSV and generated Parquet files. Local users should complete `aws login` in a terminal before starting Jupyter. Do not run an interactive sign-in command with `!aws` inside the notebook.

In [ ]:
%pip install -r requirements.txt

from pathlib import Path
import json
import os

import boto3  # AWS SDK for Python
import pandas as pd  # DataFrame library for tabular data
from botocore import UNSIGNED
from botocore.config import Config
from botocore.exceptions import ClientError

for folder in ("data/raw", "data/curated", "outputs"):
    Path(folder).mkdir(parents=True, exist_ok=True)

## 2. Choose the sample and check your identity

The default sample uses three cities and five complete years. That produces 15 small source files.

Your decision: keep the defaults for comparable workshop results, or remove one city and predict how the row and file counts will change.

In [ ]:
AWS_PROFILE = os.getenv("AWS_PROFILE") or None
AWS_REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
SOURCE_BUCKET = "noaa-gsod-pds"

YEARS = list(range(2020, 2025))
STATIONS = [
    {"station_id": "72793024233", "city": "Seattle", "city_key": "seattle"},
    {"station_id": "72530094846", "city": "Chicago", "city_key": "chicago"},
    {"station_id": "72202012839", "city": "Miami", "city_key": "miami"},
]

session_args = {"region_name": AWS_REGION}
if AWS_PROFILE:
    session_args["profile_name"] = AWS_PROFILE
# Session uses local login or SageMaker role; STS confirms the active identity.
session = boto3.Session(**session_args)

identity = session.client("sts").get_caller_identity()
print("AWS identity")
print(f"  Account ID:    {identity['Account']}")
print(f"  Principal ARN: {identity['Arn']}")
print(f"  Region:        {AWS_REGION}")
print(f"  Source files:  {len(STATIONS) * len(YEARS)}")

### Confirm permissions for your environment

**SageMaker:** Open the [IAM Roles console](https://console.aws.amazon.com/iam/home#/roles), find the execution role shown in the principal ARN above, and attach `AmazonAthenaFullAccess`. Ask the account administrator or workshop instructor if you cannot change the role.

**Local Jupyter or an IDE:** Skip the SageMaker role step. Complete `aws login` before starting the IDE. Boto3 uses the default credential chain, and the signed-in principal needs permission to create and use the project S3 bucket, Glue catalog, and Athena workgroup.

## 3. Create the AWS resources

Provided AWS plumbing: run this cell and focus on the three SDK clients and the resources they create. The bucket name contains `sagemaker` so the standard SageMaker policy can use its objects. New S3 buckets already block public access and encrypt new objects with SSE-S3.

In [ ]:
account_id = identity["Account"]
region_key = AWS_REGION.replace("_", "-")
bucket_name = f"sagemaker-noaa-weather-{account_id}-{region_key}"
database_name = f"noaa_weather_{account_id}"
workgroup_name = f"noaa-weather-{account_id}-{region_key}"

# Service clients expose the API for each AWS service.
s3 = session.client("s3")
glue = session.client("glue")
athena = session.client("athena")

# Create or reuse the project bucket.
try:
    s3.head_bucket(Bucket=bucket_name)
    print(f"Using existing bucket: {bucket_name}")
except ClientError as error:
    code = error.response["Error"].get("Code", "")
    if code not in {"404", "NoSuchBucket", "NotFound"}:
        raise
    create_args = {"Bucket": bucket_name}
    if AWS_REGION != "us-east-1":
        create_args["CreateBucketConfiguration"] = {"LocationConstraint": AWS_REGION}
    s3.create_bucket(**create_args)
    print(f"Created bucket: {bucket_name}")

# Create or reuse the Glue database.
try:
    glue.create_database(DatabaseInput={
        "Name": database_name,
        "Description": "NOAA weather personal project",
    })
    print(f"Created Glue database: {database_name}")
except glue.exceptions.AlreadyExistsException:
    print(f"Using existing Glue database: {database_name}")

# Create or update the Athena workgroup and its 100 MB scan limit.
workgroup_configuration = {
    "ResultConfiguration": {
        "OutputLocation": f"s3://{bucket_name}/athena-results/",
        "EncryptionConfiguration": {"EncryptionOption": "SSE_S3"},
    },
    "EnforceWorkGroupConfiguration": True,
    "PublishCloudWatchMetricsEnabled": True,
    "BytesScannedCutoffPerQuery": 100_000_000,
}
try:
    athena.get_work_group(WorkGroup=workgroup_name)
    athena.update_work_group(
        WorkGroup=workgroup_name,
        State="ENABLED",
        ConfigurationUpdates={
            "ResultConfigurationUpdates": workgroup_configuration["ResultConfiguration"],
            "EnforceWorkGroupConfiguration": True,
            "PublishCloudWatchMetricsEnabled": True,
            "BytesScannedCutoffPerQuery": 100_000_000,
            "RemoveBytesScannedCutoffPerQuery": False,
        },
    )
    print(f"Updated Athena workgroup: {workgroup_name}")
except athena.exceptions.InvalidRequestException:
    athena.create_work_group(
        Name=workgroup_name,
        Description="NOAA weather personal project",
        Configuration=workgroup_configuration,
    )
    print(f"Created Athena workgroup: {workgroup_name}")

project_config = {
    "account_id": account_id,
    "region": AWS_REGION,
    "bucket": bucket_name,
    "database": database_name,
    "workgroup": workgroup_name,
    "years": YEARS,
    "stations": STATIONS,
}
Path("project_config.json").write_text(json.dumps(project_config, indent=2))

## 4. Copy and inspect the NOAA files

The NOAA bucket permits unsigned reads. The loop requests only the station and year keys you selected, keeps a local copy, and uploads the unchanged bytes to your `raw/` S3 prefix.

Before running it, calculate the expected number of files from `len(STATIONS) * len(YEARS)`.

In [ ]:
public_s3 = boto3.client(
    "s3",
    region_name="us-east-1",
    config=Config(signature_version=UNSIGNED),  # Public S3 read needs no signature.
)

manifest = []
frames = []
for station in STATIONS:
    for year in YEARS:
        source_key = f"{year}/{station['station_id']}.csv"
        csv_bytes = public_s3.get_object(
            Bucket=SOURCE_BUCKET,
            Key=source_key,
        )["Body"].read()

        local_path = Path("data/raw") / str(year) / f"{station['station_id']}.csv"
        local_path.parent.mkdir(parents=True, exist_ok=True)
        local_path.write_bytes(csv_bytes)

        target_key = f"raw/year={year}/station_id={station['station_id']}/data.csv"
        s3.put_object(
            Bucket=bucket_name,
            Key=target_key,
            Body=csv_bytes,
            ContentType="text/csv",
            Metadata={"source-key": source_key, "city-key": station["city_key"]},
        )

        # Load each CSV as a DataFrame; pd.concat below combines them.
        frames.append(pd.read_csv(
            local_path,
            dtype={"STATION": "string", "FRSHTT": "string"},
        ))
        manifest.append({
            "city": station["city"],
            "year": year,
            "bytes": len(csv_bytes),
            "s3_key": target_key,
        })

raw = pd.concat(frames, ignore_index=True)
manifest_frame = pd.DataFrame(manifest)
print(f"Copied {len(manifest_frame)} files with {len(raw):,} total rows")

preview_columns = ["STATION", "DATE", "NAME", "TEMP", "SLP", "PRCP", "SNDP"]
raw[preview_columns].head(8)

## 5. Find the values that are not weather

A sentinel value is a number used to mean "measurement unavailable." It is not a real weather observation. NOAA uses numeric sentinels because the source format expects a number in each measurement field.

Examples in this dataset:

```text
TEMP, MAX, MIN, or SLP = 9999.9  -> measurement unavailable
PRCP                   = 99.99   -> precipitation unavailable
SNDP                    = 999.9   -> snow depth unavailable
```

For example, `SNDP = 999.9` does not mean 999.9 inches of snow. The next section replaces these codes with pandas null values so they are excluded from averages. The raw CSV files remain unchanged.

Exercise: inspect the output below. Which column has the most sentinel rows?

In [ ]:
sentinel_values = {
    "TEMP": 9999.9,
    "MAX": 9999.9,
    "MIN": 9999.9,
    "SLP": 9999.9,
    "PRCP": 99.99,
    "SNDP": 999.9,
}

sentinel_report = pd.DataFrame([
    {
        "column": column,
        "sentinel": sentinel,
        "sentinel_rows": int(raw[column].eq(sentinel).sum()),
        "raw_max": raw[column].max(),
        "raw_mean": raw[column].mean(),
    }
    for column, sentinel in sentinel_values.items()
])
sentinel_report

## 6. Clean the sentinel values

Cleaning changes known missing-value codes to null. Nulls are excluded from averages, while the raw files stay unchanged in `raw/`.

Exercise: comment out the `SNDP` rule, rerun the cell, and compare its mean. Restore the rule before continuing.

In [ ]:
raw_clean = raw.copy()

numeric_columns = ["LATITUDE", "LONGITUDE", "TEMP", "MAX", "MIN", "SLP", "PRCP", "SNDP"]
for column in numeric_columns:
    raw_clean[column] = pd.to_numeric(raw_clean[column], errors="coerce")

before_cleaning = {
    column: raw_clean[column].mean()
    for column in sentinel_values
}
for column, sentinel in sentinel_values.items():
    # Replace the sentinel with a null while keeping the row.
    raw_clean[column] = raw_clean[column].mask(raw_clean[column].eq(sentinel))

after_cleaning = {
    column: raw_clean[column].mean()
    for column in sentinel_values
}

pd.DataFrame({
    "mean_before": before_cleaning,
    "mean_after": after_cleaning,
    "null_rows_after": {
        column: int(raw_clean[column].isna().sum())
        for column in sentinel_values
    },
}).round(2)

## 7. Build the curated columns

The curated layer keeps fields needed for analysis and adds two business rules.

Your decision: `90 F` defines a hot day and `32 F` defines a freezing day. Change a threshold and predict which city will be affected most.

In [ ]:
HOT_DAY_F = 90
FREEZING_DAY_F = 32

column_names = {
    "STATION": "station_id",
    "DATE": "date",
    "LATITUDE": "latitude",
    "LONGITUDE": "longitude",
    "NAME": "station_name",
    "TEMP": "temp_f",
    "MAX": "max_temp_f",
    "MIN": "min_temp_f",
    "PRCP": "precipitation_in",
    "SNDP": "snow_depth_in",
}
curated = raw_clean[list(column_names)].rename(columns=column_names).copy()
curated["date"] = pd.to_datetime(curated["date"], errors="coerce")

city_by_station = {station["station_id"]: station["city"] for station in STATIONS}
city_key_by_station = {station["station_id"]: station["city_key"] for station in STATIONS}
curated["city"] = curated["station_id"].map(city_by_station)
curated["city_key"] = curated["station_id"].map(city_key_by_station)
curated["year"] = curated["date"].dt.year.astype("Int64")
curated["month"] = curated["date"].dt.month.astype("Int64")
curated["is_hot_day"] = curated["max_temp_f"].ge(HOT_DAY_F).fillna(False)
curated["is_freezing_day"] = curated["min_temp_f"].le(FREEZING_DAY_F).fillna(False)
curated = curated.sort_values(["station_id", "date"]).reset_index(drop=True)

curated[[
    "city", "date", "temp_f", "max_temp_f", "min_temp_f",
    "precipitation_in", "is_hot_day", "is_freezing_day",
]].head(8)

## 8. Validate before writing

Validation does not remove rows. It stops the pipeline when an assumption is false.

Exercise: after the checks pass, temporarily run `curated.loc[0, "temp_f"] = 200` and rerun this cell. Read the failure, then rerun the previous cell to restore the data.

In [ ]:
expected_days = sum(
    pd.Timestamp(year=year, month=12, day=31).dayofyear
    for year in YEARS
)
expected_rows = expected_days * len(STATIONS)

# Assertions stop the pipeline before invalid data is written.
assert len(curated) == expected_rows, "A station-year file has missing daily rows"
assert not curated.duplicated(["station_id", "date"]).any(), "Duplicate station dates found"
assert curated["date"].notna().all(), "A date could not be parsed"
assert curated["temp_f"].dropna().between(-150, 150).all(), "Temperature outside expected range"

quality_summary = pd.DataFrame([{
    "rows": len(curated),
    "expected_rows": expected_rows,
    "stations": curated["station_id"].nunique(),
    "duplicate_dates": curated.duplicated(["station_id", "date"]).sum(),
    "missing_temp_pct": round(curated["temp_f"].isna().mean() * 100, 2),
    "missing_snow_depth_pct": round(curated["snow_depth_in"].isna().mean() * 100, 2),
}])
print("Quality checks passed")
quality_summary

## 9. Write Parquet and compare storage

Parquet stores typed columns and compresses repeated values. The output is partitioned by city and year so Athena can skip unrelated folders.

Before running the cell, predict whether 15 Parquet files will use more or fewer bytes than the 15 raw CSV files.

In [ ]:
curated_manifest = []
for (city_key, year), partition in curated.groupby(["city_key", "year"], observed=True):
    city_key = str(city_key)
    year = int(str(year))
    parquet_frame = partition.drop(columns=["city_key", "year"]).copy()
    parquet_frame["date"] = parquet_frame["date"].dt.date

    local_path = Path("data/curated") / f"city={city_key}" / f"year={year}" / "weather.parquet"
    local_path.parent.mkdir(parents=True, exist_ok=True)
    # Parquet stores typed columns and compresses repeated values.
    parquet_frame.to_parquet(local_path, index=False, compression="snappy")

    target_key = f"curated/city={city_key}/year={year}/weather.parquet"
    s3.upload_file(str(local_path), bucket_name, target_key)
    curated_manifest.append({
        "city_key": city_key,
        "year": year,
        "rows": len(parquet_frame),
        "bytes": local_path.stat().st_size,
        "s3_key": target_key,
    })

curated_manifest_frame = pd.DataFrame(curated_manifest)
raw_bytes = int(manifest_frame["bytes"].sum())
parquet_bytes = int(curated_manifest_frame["bytes"].sum())
storage_comparison = pd.DataFrame([
    {"format": "Raw CSV", "files": len(manifest_frame), "bytes": raw_bytes},
    {"format": "Snappy Parquet", "files": len(curated_manifest_frame), "bytes": parquet_bytes},
])
storage_comparison["megabytes"] = (storage_comparison["bytes"] / 1_000_000).round(3)
storage_comparison

## 10. Register the S3 data in AWS Glue

Provided catalog plumbing: these definitions tell Athena the column types, S3 locations, and partition keys. Focus on `Columns`, `Location`, and `PartitionKeys`; you do not need to memorize the Hadoop format class names.

In [ ]:
# Glue records schemas and projected S3 partitions; it does not move data.
raw_columns = [
    {"Name": column.lower(), "Type": "string"}
    for column in raw.columns
]
curated_types = {
    "station_id": "string",
    "date": "date",
    "latitude": "double",
    "longitude": "double",
    "station_name": "string",
    "temp_f": "double",
    "max_temp_f": "double",
    "min_temp_f": "double",
    "precipitation_in": "double",
    "snow_depth_in": "double",
    "city": "string",
    "month": "bigint",
    "is_hot_day": "boolean",
    "is_freezing_day": "boolean",
}
curated_columns = [
    {"Name": name, "Type": data_type}
    for name, data_type in curated_types.items()
]

station_values = ",".join(station["station_id"] for station in STATIONS)
city_key_values = ",".join(station["city_key"] for station in STATIONS)
year_range = f"{min(YEARS)},{max(YEARS)}"

raw_table = {
    "Name": "gsod_raw",
    "Description": "Unchanged NOAA GSOD CSV files",
    "TableType": "EXTERNAL_TABLE",
    "Parameters": {
        "classification": "csv",
        "skip.header.line.count": "1",
        "projection.enabled": "true",
        "projection.year.type": "integer",
        "projection.year.range": year_range,
        "projection.station_id.type": "enum",
        "projection.station_id.values": station_values,
        "storage.location.template": f"s3://{bucket_name}/raw/year=${{year}}/station_id=${{station_id}}/",
    },
    "PartitionKeys": [
        {"Name": "year", "Type": "int"},
        {"Name": "station_id", "Type": "string"},
    ],
    "StorageDescriptor": {
        "Columns": raw_columns,
        "Location": f"s3://{bucket_name}/raw/",
        "InputFormat": "org.apache.hadoop.mapred.TextInputFormat",
        "OutputFormat": "org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat",
        "SerdeInfo": {
            "SerializationLibrary": "org.apache.hadoop.hive.serde2.OpenCSVSerde",
            "Parameters": {"separatorChar": ",", "quoteChar": '"'},
        },
    },
}

curated_table = {
    "Name": "weather_curated",
    "Description": "Cleaned NOAA observations in partitioned Parquet",
    "TableType": "EXTERNAL_TABLE",
    "Parameters": {
        "classification": "parquet",
        "projection.enabled": "true",
        "projection.city_key.type": "enum",
        "projection.city_key.values": city_key_values,
        "projection.year.type": "integer",
        "projection.year.range": year_range,
        "storage.location.template": f"s3://{bucket_name}/curated/city=${{city_key}}/year=${{year}}/",
    },
    "PartitionKeys": [
        {"Name": "city_key", "Type": "string"},
        {"Name": "year", "Type": "int"},
    ],
    "StorageDescriptor": {
        "Columns": curated_columns,
        "Location": f"s3://{bucket_name}/curated/",
        "InputFormat": "org.apache.hadoop.hive.ql.io.parquet.MapredParquetInputFormat",
        "OutputFormat": "org.apache.hadoop.hive.ql.io.parquet.MapredParquetOutputFormat",
        "SerdeInfo": {
            "SerializationLibrary": "org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe",
            "Parameters": {"serialization.format": "1"},
        },
    },
}

def upsert_table(table_input):
    table_name = table_input["Name"]
    try:
        glue.get_table(DatabaseName=database_name, Name=table_name)
    except glue.exceptions.EntityNotFoundException:
        glue.create_table(DatabaseName=database_name, TableInput=table_input)
        return "created"
    glue.update_table(DatabaseName=database_name, TableInput=table_input)
    return "updated"

catalog_actions = {
    "gsod_raw": upsert_table(raw_table),
    "weather_curated": upsert_table(curated_table),
}
print(json.dumps(catalog_actions, indent=2))

## 11. Save the handoff to Notebook 2

Notebook 2 needs only the resource names and selected sample. The generated configuration contains no credentials.

In [ ]:
project_config.update({
    "raw_objects": len(manifest_frame),
    "raw_bytes": raw_bytes,
    "curated_objects": len(curated_manifest_frame),
    "curated_bytes": parquet_bytes,
    "rows": len(curated),
})
Path("project_config.json").write_text(json.dumps(project_config, indent=2))

print("Pipeline ready")
print(f"  Raw data:      s3://{bucket_name}/raw/")
print(f"  Curated data:  s3://{bucket_name}/curated/")
print(f"  Rows:          {len(curated):,}")
print(f"  Next notebook: 02_query_and_visualize.ipynb")

## 12. View what you built in the AWS console

Open each resource and connect the notebook code to what AWS created:

1. In S3, compare the unchanged CSV objects under `raw/` with the Parquet objects under `curated/`.
2. In Glue, open `gsod_raw` and `weather_curated`. Compare their column types and partition keys.
3. In Athena, open the project workgroup and find its result location and 100 MB scan limit.

Jupyter usually makes the printed URLs clickable. If it does not, copy a URL into a browser.

In [ ]:
console_urls = {
    "S3 raw objects": (
        f"https://{AWS_REGION}.console.aws.amazon.com/s3/buckets/{bucket_name}"
        f"?region={AWS_REGION}&bucketType=general&prefix=raw%2F&tab=objects"
    ),
    "S3 curated objects": (
        f"https://{AWS_REGION}.console.aws.amazon.com/s3/buckets/{bucket_name}"
        f"?region={AWS_REGION}&bucketType=general&prefix=curated%2F&tab=objects"
    ),
    "Glue database": (
        f"https://{AWS_REGION}.console.aws.amazon.com/glue/home"
        f"?region={AWS_REGION}#/v2/data-catalog/databases/view/{database_name}"
    ),
    "Athena workgroups": (
        f"https://{AWS_REGION}.console.aws.amazon.com/athena/home"
        f"?region={AWS_REGION}#/workgroups"
    ),
}

for label, url in console_urls.items():
    print(label)
    print(url)
    print()

print(f"Athena workgroup to open: {workgroup_name}")

## What you learned

You saw why numeric sentinels need domain knowledge, separated cleaning from validation, turned raw CSV into typed Parquet, and registered both layouts for Athena. Notebook 2 measures the query scan difference and builds the charts.